# 05 — HSV12 + HOG-PCA + SVM

Hai khối feature đã L2 được nối theo thứ tự HSV12 rồi HOG. Không thêm trọng số và không chuẩn hóa lại sau nối. Kết quả đánh giá một cách ghép cố định, không đại diện cho mọi cách kết hợp màu và đường nét.


In [ ]:
from pathlib import Path
import os, sys
# Bỏ comment nếu Kaggle gắn nhiều bộ code hoặc dữ liệu.
# os.environ['CS231_PROJECT_ROOT'] = '/kaggle/input/TEN-DATASET-CODE'
# os.environ['CS231_DATA_ROOT'] = '/kaggle/input/TEN-DATASET-ANH/DATA'
# os.environ['CS231_SVM_PREVIOUS_RESULTS'] = '/kaggle/input/.../CS231_SVM/results_224'
os.environ.setdefault('MPLCONFIGDIR', '/tmp/cs231-matplotlib')
roots = [Path.cwd(), *Path.cwd().parents]
roots += [p for pattern in ('*', 'datasets/*/*') for p in Path('/kaggle/input').glob(pattern)]
if os.environ.get('CS231_PROJECT_ROOT'):
    roots = [Path(os.environ['CS231_PROJECT_ROOT'])]
projects = sorted({p.resolve() for p in roots if (p/'src/preprocessing.py').is_file()})
if len(projects) != 1:
    raise FileNotFoundError('Đặt CS231_PROJECT_ROOT trỏ tới thư mục chứa src.')
PROJECT_ROOT = projects[0]
sys.path.insert(0, str(PROJECT_ROOT))
from src.experiments.runtime import setup, restore_previous_output
DATA_ROOT, RESULTS_ROOT = setup('SVM', PROJECT_ROOT)

In [ ]:
restore_previous_output(RESULTS_ROOT, 'SVM', 'hsv12_hog/svm_grid_dev.csv')

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.SVM.pipeline import (
    CLASSES, FEATURE_FOLDERS, manifest_for_split, extract_matrix, l2,
    save_features, load_features, same_manifest, run_dev_grid, save_dev_result,
    save_json, hsv12, hog_raw,
)
from src.preprocessing import read_rgb224

## 1. Đọc cache và đối chiếu thứ tự ảnh

Mỗi hàng của hai nhánh phải trỏ đến cùng đường dẫn và nhãn trước khi nối vector.


In [ ]:
HSV_DIR = RESULTS_ROOT / 'hsv12'
HOG_DIR = RESULTS_ROOT / 'hog_pca'
OUTPUT = RESULTS_ROOT / 'hsv12_hog_pca'
FEATURE_ID = 'HSV12_HOG_PCA'
FEATURE_PIPELINE = 'concatenate L2 HSV12 then L2 hog_pca; no post-concat normalization'
X_train_hsv, y_train, train_manifest = load_features(HSV_DIR, 'train')
X_dev_hsv, y_dev, dev_manifest = load_features(HSV_DIR, 'dev')
X_train_hog, hog_y_train, hog_train_manifest = load_features(HOG_DIR, 'train')
X_dev_hog, hog_y_dev, hog_dev_manifest = load_features(HOG_DIR, 'dev')
same_manifest(train_manifest, hog_train_manifest)
same_manifest(dev_manifest, hog_dev_manifest)
if not np.array_equal(y_train, hog_y_train) or not np.array_equal(y_dev, hog_y_dev):
    raise ValueError('Nhãn hai nhánh không khớp')

## 2. Ghép feature và lưu cache

Nếu hai khối đều có norm 1, norm của vector ghép gần căn 2. Euclidean trên vector ghép cộng đóng góp khoảng cách bình phương từ từng khối.


In [ ]:
X_train = np.concatenate([X_train_hsv, X_train_hog], axis=1).astype(np.float32)
X_dev = np.concatenate([X_dev_hsv, X_dev_hog], axis=1).astype(np.float32)
save_features(OUTPUT, train_manifest, dev_manifest, X_train, X_dev)
print('Train/dev:', X_train.shape, X_dev.shape)
print('Norm vector ghép:', np.linalg.norm(X_train[0]))

## 3. Khảo sát SVC trên dev


In [ ]:
grid, best_pred, best_params = run_dev_grid(X_train, y_train, X_dev, y_dev)
scores = save_dev_result(OUTPUT, FEATURE_ID, grid, best_pred, dev_manifest)
save_json(OUTPUT / 'run_config.json', {
    'feature_id': FEATURE_ID, 'feature_dim': int(X_train.shape[1]),
    'preprocessing': 'RGB uint8 224x224 OpenCV INTER_LINEAR',
    'feature_pipeline': FEATURE_PIPELINE,
    'svm_grid': 'linear C=(0.1,1,10); rbf C=(1,10), gamma=(scale,10)',
    'class_weight': None, 'cache_size_mb': 512,
    'decision_function_shape': 'ovr', 'break_ties': True,
    'best_params': best_params, 'fit_split': 'train', 'selection_split': 'dev',
})
print('Cấu hình thắng trên dev:')
display(grid.head(7))
print('Accuracy:', scores['accuracy'], 'Macro-F1:', scores['macro_f1'],
      'MAE VND/ảnh:', scores['mae_vnd'])
display(pd.DataFrame(scores['per_class'])[
    ['denomination_vnd', 'support', 'precision', 'recall', 'f1']])

## 4. Ma trận nhầm lẫn và so sánh nhánh riêng


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(np.asarray(scores['confusion_matrix']),
                       display_labels=CLASSES).plot(ax=ax, cmap='Blues',
                                                     values_format='d', xticks_rotation=45)
ax.set_title(f'{FEATURE_ID} + SVM — dev')
plt.tight_layout()
plt.savefig(OUTPUT / 'confusion_matrix_dev.png', dpi=180)
plt.savefig(OUTPUT / 'confusion_matrix_dev.pdf', dpi=180)
plt.show()

In [ ]:
hsv_best = pd.read_csv(HSV_DIR / 'svm_best_dev.csv')
hog_best = pd.read_csv(HOG_DIR / 'svm_best_dev.csv')
fusion_best = pd.read_csv(OUTPUT / 'svm_best_dev.csv')
display(pd.concat([hsv_best, hog_best, fusion_best], ignore_index=True)[
    ['feature_id', 'kernel', 'C', 'gamma', 'accuracy', 'macro_f1', 'mae_vnd']])